#### dissert — Kaggle worker template

Generic single-config launch template for XDash's Kaggle runner (EXPERIMENT_AUTOMATION_PLAN.md
§2.4/§2.5). **Do not edit or push this notebook directly** — the dashboard renders a copy of it
per launch, substituting the `# DASHBOARD:LAUNCH_SPEC` cell below, then pushes the rendered copy.
One push = one config, running train then eval sequentially inside this one kernel (a Kaggle
account only runs one kernel at a time, so there is no separate train-only/eval-only push to
chain a second half onto).

Environment setup (apt python3.8, a venv, pinned torch/mmcv wheels) and the private-repo clone
via a Kaggle secret are carried over from this repo's existing `notebooks/kaggle_run.ipynb`,
which is proven to work on this repo — this template does not reinvent that part, only adds the
dashboard integration (LAUNCH_SPEC marker, dataset auto-attach, two-stage run, result packaging)
around it.

**Not yet verified against a real Kaggle push** (unlike segpriors' template, which was). Verify
end-to-end before relying on it for an unattended batch.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
print(subprocess.run(["python3", "--version"], capture_output=True, text=True).stdout)


## 2. Clone the repo

Private repo: `GITHUB_TOKEN` must be attached as a Kaggle secret on this notebook
(Add-ons -> Secrets) — same as the existing `notebooks/kaggle_run.ipynb`.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("GITHUB_TOKEN")

username = "Syfur007"
repo_name = "dissert"

%cd /kaggle/working
if os.path.exists(repo_name):
    os.system(f"rm -rf {repo_name}")

!git clone https://{token}@github.com/{username}/{repo_name}.git
%cd {repo_name}


## 3. Reproduce the training environment

In [ ]:
!add-apt-repository ppa:deadsnakes/ppa -y
!apt-get update -qq
!apt-get install -y python3.8 python3.8-venv python3.8-dev
!python3.8 -m venv /kaggle/working/py38_env
PY = "/kaggle/working/py38_env/bin/python"
PIP = "/kaggle/working/py38_env/bin/pip"


In [ ]:
# --only-binary/--prefer-binary: falling back to a source build here can silently burn
# 20-40+ minutes of this notebook's own time budget (see EXPERIMENT_AUTOMATION_PLAN.md §8.1 —
# this setup time comes out of the same session as training and must be reserved for).
!{PIP} install --upgrade pip
!{PIP} install --only-binary=:all: SimpleITK==2.2.1
!{PIP} install --prefer-binary torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
!{PIP} install --prefer-binary mmcv-full -f https://download.openmmlab.com/mmcv/dist/cu117/torch1.13.0/index.html
!{PIP} install --prefer-binary -r requirements.txt


## 4. Launch spec — filled in by the dashboard

Left as-is (unsubstituted), the next cell fails loudly instead of silently running nothing.

In [ ]:
# DASHBOARD:LAUNCH_SPEC — substituted by backend/kaggle.py's _render_launch_notebook() before
# push. Do not hand-edit these values in this template file — edit a *pushed copy* only if
# you're debugging a specific run, never this shared template.
CONFIG_PATH = "__DASHBOARD_CONFIG_PATH__"
EXTRA_ARGS = "__DASHBOARD_EXTRA_ARGS__"
RUN_MODE = "__DASHBOARD_RUN_MODE__"
RESUME_FROM_WORKER_ID = "__DASHBOARD_RESUME_FROM_WORKER_ID__"
RESUME_FROM_RUN_ID = "__DASHBOARD_RESUME_FROM_RUN_ID__"
RESUME_FROM_RESULTS_DIR = "__DASHBOARD_RESUME_FROM_RESULTS_DIR__"
RESUME_FROM_MANIFEST_PATH = "__DASHBOARD_RESUME_FROM_MANIFEST_PATH__"
CHAIN_ID = "__DASHBOARD_CHAIN_ID__"
VERSION_INDEX = "__DASHBOARD_VERSION_INDEX__"
# Resolved server-side from this deployment's repo profile (repos/dissert.yaml's train_script/
# eval_script) — dissert's own train.py takes --seed/--seeds directly (see repos/dissert.yaml's
# seed_arg), so no sweep wrapper is needed the way segpriors needs one.
TRAIN_SCRIPT = "__DASHBOARD_TRAIN_SCRIPT__"
EVAL_SCRIPT = "__DASHBOARD_EVAL_SCRIPT__"
# Flags always appended to the eval command only — repos/dissert.yaml's eval_default_args is
# currently [] (see that file's own note: --ensemble there means fold-ensemble, not confirmed
# as a sane default yet), so this is normally empty for this repo.
EVAL_EXTRA_FLAGS = "__DASHBOARD_EVAL_EXTRA_FLAGS__"

assert CONFIG_PATH and not CONFIG_PATH.startswith("__DASHBOARD_"), (
    "CONFIG_PATH was never substituted -- this notebook was uploaded/run directly instead of "
    "pushed through the dashboard's Kaggle launch flow, which is what fills in this cell."
)
if RUN_MODE == "resume":
    assert RESUME_FROM_RUN_ID and RESUME_FROM_RESULTS_DIR, (
        "Resume run requires RESUME_FROM_RUN_ID and RESUME_FROM_RESULTS_DIR; this worker was "
        "launched in resume mode without the required metadata."
    )
    assert not RESUME_FROM_RESULTS_DIR.startswith("__DASHBOARD_"), (
        "RESUME_FROM_RESULTS_DIR was never substituted -- resume metadata is incomplete."
    )
print("CONFIG_PATH:", CONFIG_PATH)
print("EXTRA_ARGS:", EXTRA_ARGS or "(none)")
print("RUN_MODE:", RUN_MODE)
print("RESUME_FROM_RUN_ID:", RESUME_FROM_RUN_ID or "(none)")
print("RESUME_FROM_RESULTS_DIR:", RESUME_FROM_RESULTS_DIR or "(none)")
print("TRAIN_SCRIPT:", TRAIN_SCRIPT)
print("EVAL_SCRIPT:", EVAL_SCRIPT)
print("EVAL_EXTRA_FLAGS:", EVAL_EXTRA_FLAGS or "(none)")


## 5. Attach the dataset this config needs

Generic by design: resolves `dataset.name`/`dataset.root` from the *composed* config the same
way `utils.config.load_config()` does for every other entry point, then looks for a same-named
directory already attached under `/kaggle/input/` and symlinks it into place. `load_config()`
returns a plain dict here (dissert's own schema — see `utils/config.py`, `orchestration/
schema.py`), unlike segpriors' pydantic-object config, so this indexes with `["dataset"]["root"]`
rather than attribute access. If nothing matches, attach the right Kaggle dataset via
**+ Add Data** first — check `configs/dataset/*.yaml` for what root layout that dataset expects.

In [ ]:
import subprocess, glob, shutil, json as _json

os.environ["PYTHONPATH"] = os.getcwd()
probe = subprocess.run(
    [PY, "-c",
     "from utils.config import load_config; import json; c = load_config(" + repr(CONFIG_PATH) + "); "
     "print(json.dumps({'name': c['dataset']['name'], 'root': c['dataset']['root']}))"],
    capture_output=True, text=True,
)
assert probe.returncode == 0, f"Could not resolve dataset info for {CONFIG_PATH}:\n{probe.stderr}"
ds = _json.loads(probe.stdout.strip().splitlines()[-1])
dataset_name, dataset_root = ds["name"], ds["root"]
print("dataset:", dataset_name, "-> root:", dataset_root)

if not os.path.isdir(dataset_root):
    leaf = os.path.basename(dataset_root.rstrip("/"))
    candidates = [p for p in glob.glob(f"/kaggle/input/**/{leaf}", recursive=True) if os.path.isdir(p)]
    assert candidates, (
        f"No '{leaf}' directory found under /kaggle/input/ for dataset '{dataset_name}' — "
        f"attach the matching Kaggle dataset via + Add Data first."
    )
    os.makedirs(os.path.dirname(dataset_root) or ".", exist_ok=True)
    os.symlink(candidates[0], dataset_root)
    print("symlinked", candidates[0], "->", dataset_root)


## 6. Run

Train, then eval — both inside this one kernel execution. Eval only runs if train exits 0,
mirroring the local scheduler's own skip-on-failure semantics for a `mode="both"` launch
(EXPERIMENT_AUTOMATION_PLAN.md §2.4).

In [ ]:
import shlex

cmd_train = [PY, TRAIN_SCRIPT, "--config", CONFIG_PATH] + (shlex.split(EXTRA_ARGS) if EXTRA_ARGS else [])
print("Running (train):", " ".join(shlex.quote(c) for c in cmd_train))
result = subprocess.run(cmd_train)
assert result.returncode == 0, f"{TRAIN_SCRIPT} exited with code {result.returncode}"

eval_extra_flags = shlex.split(EVAL_EXTRA_FLAGS) if EVAL_EXTRA_FLAGS else []
cmd_eval = [PY, EVAL_SCRIPT, "--config", CONFIG_PATH] + eval_extra_flags + (shlex.split(EXTRA_ARGS) if EXTRA_ARGS else [])
print("Running (eval):", " ".join(shlex.quote(c) for c in cmd_eval))
result = subprocess.run(cmd_eval)
assert result.returncode == 0, f"{EVAL_SCRIPT} exited with code {result.returncode}"


## 7. Package results for download

dissert's own layout (OUTPUT_LAYOUT.md, `manifest_layout: "experiments"`): everything lives
under `outputs/` (nested per-experiment dirs plus the flat `outputs/ledger/`), unlike segpriors'
`artifacts/`+`logs/`+`checkpoints/` split — `backend/kaggle.py`'s `register_ledger()` and
`usage_history()` both branch on `manifest_layout` to find it at this path inside the
downloaded zip.

In [ ]:
!cd /kaggle/working/dissert && zip -qr /kaggle/working/results.zip outputs/ -x "*.pth"
!cd /kaggle/working/dissert && find outputs -name "*.pth" | zip -q /kaggle/working/checkpoints.zip -@
!ls -lh /kaggle/working/*.zip
